# Script to run GPT using the batch API

In [55]:
import json
import os
import random

import tiktoken
from openai import OpenAI

from scripts.prompting.prompts import task_description_v2
from scripts.prompting.run_llms import open_input_file, get_input_text

In [2]:
def create_batch_request(prompt, request_id, model_id):
    return {
        "custom_id": request_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model_id,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "max_tokens": 4096
        }
    }


In [74]:
def prepare_instructions(train_folder, dot_train_data, number_of_examples=1, sel_random=False):
    examples = list()
    if sel_random:
        file1 = random.choice(os.listdir(train_folder))
        print(f'Randomly Selected File: {file1}')
        data = open_input_file(f'{train_folder}/{file1}')
        intput_text = get_input_text(data)
        output_example = dot_train_data[file1]['target']
        examples.append((intput_text, output_example))
    else:
        for file1 in os.listdir(train_folder):
            if len(examples) < number_of_examples:
                print(f'Sequentially Selected File: {file1}')
                data = open_input_file(f'{train_folder}/{file1}')
                intput_text = get_input_text(data)
                output_example = dot_train_data[file1]['target']
                examples.append((intput_text, output_example))

    instruct_examples = list()
    for i, example in enumerate(examples):
        input_expl = f'Example {i + 1}:\n' + example[0] + '\n'
        output_expl = "Expected output:\n" + example[1] + "\n"
        instruct_examples.append(input_expl + output_expl)

    return "\n".join(instruct_examples)

In [57]:
def from_dataset_to_batch_req(test_folder, train_folder, dot_train_data, instructions_func, output_file, num_of_files, num_of_examples, sel_random, model_id):
    enc = tiktoken.encoding_for_model(model_id)
    json_lines = list()

    examples = prepare_instructions(train_folder, dot_train_data, num_of_examples, sel_random)
    final_instructions = instructions_func(examples)

    total_tokens = 0
    count = 0
    for i, file1 in enumerate(os.listdir(test_folder)):
        if count == num_of_files:
            break

        count += 1
        data = open_input_file(f'{test_folder}/{file1}')
        text = get_input_text(data)
        prompt = final_instructions + '\n' + text

        # Count the number of tokens in the prompt
        encoded_text = enc.encode(prompt)
        total_tokens += len(encoded_text)
        print(f'Number of tokens in prompt={len(encoded_text)}')

        req = create_batch_request(prompt, file1, model_id)
        json_lines.append(req)
    
    print(f'Total number of tokens in all prompts={total_tokens}')
    split = 1
    if total_tokens > 90000: 
        if total_tokens/2 < 90000:
            split = 2
        elif total_tokens/3 < 90000:
            split = 3
        else: 
            split = 4
    
    chunk_size = len(json_lines) // split
    chunks = [json_lines[i:i + chunk_size] for i in range(0, len(json_lines), chunk_size)]
    for i in range(len(chunks)):
        with open(f'{output_file}_chunk{i}.jsonl', 'w', encoding='utf8') as file1:
            for req in chunks[i]:
                file1.write(json.dumps(req) + '\n')


In [58]:
def upload_batch_request(client, input_request_jsonl):
    batch_input_file = client.files.create(
        file=open(input_request_jsonl, 'rb'),
        purpose='batch'
    )

    print(f'Batch input file with id {batch_input_file.id} uploaded')
    return batch_input_file.id


In [59]:
def run_batch(client, batch_input_file_id):
    create = client.batches.create(input_file_id=batch_input_file_id, endpoint="/v1/chat/completions",
                                   completion_window="24h", metadata={"description": "eventfull gpt4o batch job"})

    print(f'Batch object created: {create}')
    return create.id


In [60]:
def check_batch_status(client, batch_id, output_file):
    batch = client.batches.retrieve(batch_id)
    print(f'Batch status: {batch}')

    if batch.status == 'completed':
        file_response = client.files.content(batch.output_file_id)
        with open(output_file, 'w', encoding='utf8') as file:
            for req in file_response.text.split('\n'):
                file.write(json.dumps(req) + '\n')
        print(file_response.text)


In [61]:
def convert_response(response_file, final_output_file):
    converted = dict()
    with (open(response_file, 'r', encoding='utf8') as file1):
        for line in file1:
            if not line.strip():
                continue

            response = json.loads(line)
            res_loads = json.loads(response)
            converted[res_loads['custom_id']] = {'target': res_loads['response']['body']['choices'][0]['message']['content']}

    with open(final_output_file, 'w', encoding='utf8') as file2:
        json.dump(converted, file2, indent=4)


Step-0: Initialize the client

In [110]:
with open('/Users/aloneirew/workspace/DenseEREGraph/data/my_data/api_key.txt', 'r') as file:
    api_key = file.readline().strip()

_client = OpenAI(api_key=api_key)

_test_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/EventFullTrainExports/test'
_train_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/EventFullTrainExports/dev'
_dot_train_data = open_input_file('/Users/aloneirew/workspace/DenseEREGraph/data/DOT_format/EventFull_dev_dot.json')

_output_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/batch_req/eventfull_request_gpt4o_1exmlps_rand_21.jsonl'
_response_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/batch_req/eventfull_response_gpt4o_1exmlps_rand_21.jsonl'
_final_output_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/batch_req/eventfull_gpt4o_1exmlps_rand_21_task_description_v2.json'

_instructions_func = task_description_v2
_num_of_files_to_prepare = -1
_random_example = True
_num_of_examples = 1
_model_id = "gpt-4o"

Step-1: Create the request and save it to a file

In [108]:
print("Running Step-1")
from_dataset_to_batch_req(_test_folder, _train_folder, _dot_train_data, _instructions_func, _output_file, _num_of_files_to_prepare, _num_of_examples, _random_example, _model_id)

Running Step-1
Randomly Selected File: 21_final.json
The mentions are-['admitted', 'complaint', 'shooting', 'receiving', 'driving', 'running', 'called', 'jumping', 'running', 'shot', 'survived', 'sent', 'cleared', 'passing', 'charged', 'appeared', 'testified']
The input text is-– Could faster action from Uber have stopped the Kalamazoo rampage ? The company has <admitted(29)> it received a <complaint(25)> about driver Jason Dalton hours before Saturday 's mass <shooting(47)> and failed to act on it , the Guardian reports . Passenger Matt Mellen may have been present at the moment the alleged gunman snapped : Mellen tells WWMT that after <receiving(15)> a phone call , Dalton started <driving(18)> like a maniac , sideswiping cars and <running(7)> stop signs . Mellen <called(34)> both 911 and Uber after <jumping(49)> out of the car and <running(41)> away . An Uber spokesperson says the complaint fell " into the back bucket " because the feedback was n't prioritized and the company prefers

Step-2: upload the request to the server

In [111]:
print("Running Step-2")
print(_output_file)
_input_request_jsonl = _output_file
_batch_input_file_id = upload_batch_request(_client, _input_request_jsonl)

Running Step-2
/Users/aloneirew/workspace/DenseEREGraph/data/my_data/batch_req/eventfull_request_gpt4o_1exmlps_rand_21.jsonl
Batch input file with id file-uzAJ4GxW938yvRB0nSgZVWxW uploaded


Step-3: Run the batch

In [112]:
print("Running Step-3")
print(_batch_input_file_id)
_batch_run_id = run_batch(_client, _batch_input_file_id)

Running Step-3
file-uzAJ4GxW938yvRB0nSgZVWxW
Batch object created: Batch(id='batch_fU7Qw3Wa3L8zcVR03FKBBlNl', completion_window='24h', created_at=1726422583, endpoint='/v1/chat/completions', input_file_id='file-uzAJ4GxW938yvRB0nSgZVWxW', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1726508983, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'eventfull gpt4o batch job'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))


Step-4: Check the status of the batch

In [114]:
print("Running Step-4")
print(_batch_run_id)
check_batch_status(_client, _batch_run_id, _response_file)

Running Step-4
batch_fU7Qw3Wa3L8zcVR03FKBBlNl
Batch status: Batch(id='batch_fU7Qw3Wa3L8zcVR03FKBBlNl', completion_window='24h', created_at=1726422583, endpoint='/v1/chat/completions', input_file_id='file-uzAJ4GxW938yvRB0nSgZVWxW', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1726423221, error_file_id=None, errors=None, expired_at=None, expires_at=1726508983, failed_at=None, finalizing_at=1726423219, in_progress_at=1726422585, metadata={'description': 'eventfull gpt4o batch job'}, output_file_id='file-T2n5hyMtNUJTam1dvIYYBlWp', request_counts=BatchRequestCounts(completed=20, failed=0, total=20))
{"id": "batch_req_sQAmCU9SR4pkjfqyyLFiSk6q", "custom_id": "10_final.json", "response": {"status_code": 200, "request_id": "96376a0deeb99bd039125818751a658c", "body": {"id": "chatcmpl-A7nncVpaYqWo5bCsrWDO4YzisjBhs", "object": "chat.completion", "created": 1726423088, "model": "gpt-4o-2024-05-13", "choices": [{"index": 0, "message": {"role": "assistant", 

Step-5: Convert response to DOT format response

In [116]:
print("Running Step-5")
convert_response(_response_file, _final_output_file)
print("Done!")

Running Step-5
Done!


Cancelling a batch

In [33]:
_client.batches.cancel("batch_xnWSUbQRSq3ZSZCwj67gtPLi")

ConflictError: Error code: 409 - {'error': {'message': "Cannot cancel a batch with status 'failed'.", 'type': 'invalid_request_error', 'param': None, 'code': None}}